# 🇮🇩 Modul 02: Analisis Tren Longitudinal & Inflasi Komoditas
### Indonesian Food Commodity Price Intelligence Platform — Inflation & Trend Analytics
**Author**: Fauzi Noorsyabani · Full Stack AI Engineer  
**Dataset**: Harga Rata-Rata 11 Komoditas Pangan (2023–2025)

---
### 🎯 Tujuan Analisis
1. Menghitung laju inflasi Year-over-Year (YoY %) untuk periode 2023→2024 dan 2024→2025.
2. Menghitung laju pertumbuhan majemuk 2-tahun (CAGR 2023–2025) untuk mengidentifikasi komoditas dengan tekanan harga terkuat.
3. Mengkorelasikan dinamika fluktuasi harga antara komoditas hortikultura (cabai, bawang) dengan komoditas ternak dan sembako primer.

In [ ]:
# Environment Setup & Library Installation
# Cell ini kompatibel untuk dijalankan di Google Colab maupun Jupyter Notebook lokal

import sys
import subprocess

# Instalasi paket jika belum tersedia (terutama di Google Colab)
required_packages = ['pandas', 'numpy', 'scikit-learn', 'matplotlib', 'seaborn', 'beautifulsoup4', 'lxml']
for pkg in required_packages:
    try:
        __import__(pkg)
    except ImportError:
        print(f"📦 Menginstal {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import os
import io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from bs4 import BeautifulSoup

# Pengaturan styling visualisasi
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#CCCCCC'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['grid.alpha'] = 0.7

print("✅ Lingkungan analisis siap digunakan (Local & Google Colab compatible)")


In [ ]:
# Fungsi Universal Pemuatan Data (Local & Colab Compatible)
def load_harga_pangan():
    # Cek lokasi file di berbagai kemungkinan path
    possible_paths = [
        "harga-rata-rata-bahan-pangan-komponen-data.xls",
        "../harga-rata-rata-bahan-pangan-komponen-data.xls",
        "data/harga_pangan_clean.csv",
        "../data/harga_pangan_clean.csv",
        "/content/harga-rata-rata-bahan-pangan-komponen-data.xls"
    ]
    
    file_path = None
    for p in possible_paths:
        if os.path.exists(p):
            file_path = p
            break
            
    if file_path is None:
        # Fallback embedded dataset jika file xls belum diunggah ke Colab
        print("⚠️ File lokal tidak ditemukan. Menggunakan embedded dataset resmi BPS/Kemendag (33 data point)...")
        raw_html = """
        <table>
          <thead>
            <tr><th><b>periode_data</b></th><th><b>komoditas</b></th><th><b>harga_rata_rata</b></th></tr>
          </thead>
          <tbody>
            <tr><td>2023</td><td>BERAS IR I</td><td>12,759.81</td></tr>
            <tr><td>2023</td><td>BERAS IR III</td><td>11,238.67</td></tr>
            <tr><td>2023</td><td>CABAI RAWIT MERHA</td><td>62,417.22</td></tr>
            <tr><td>2023</td><td>CABAI MERAH KERITING</td><td>51,684.27</td></tr>
            <tr><td>2023</td><td>BAWANG MERAH</td><td>39,974.05</td></tr>
            <tr><td>2023</td><td>BAWANG PUTIH</td><td>37,997.28</td></tr>
            <tr><td>2023</td><td>DAGING AYAM</td><td>40,142.56</td></tr>
            <tr><td>2023</td><td>TELUR AYAM</td><td>29,005.93</td></tr>
            <tr><td>2023</td><td>DAGING SAPI</td><td>140,288.21</td></tr>
            <tr><td>2023</td><td>GULA PASIR</td><td>14,954.54</td></tr>
            <tr><td>2023</td><td>MINYAK GORENG</td><td>15,777.51</td></tr>
            <tr><td>2025</td><td>BERAS IR I</td><td>15169</td></tr>
            <tr><td>2025</td><td>BERAS IR III</td><td>13599</td></tr>
            <tr><td>2025</td><td>CABAI RAWIT MERAH</td><td>70942</td></tr>
            <tr><td>2025</td><td>CABAI MERAH KERITING</td><td>59892</td></tr>
            <tr><td>2025</td><td>BAWANG MERAH</td><td>50531</td></tr>
            <tr><td>2025</td><td>BAWANG PUTIH</td><td>44513</td></tr>
            <tr><td>2025</td><td>DAGING AYAM</td><td>41071</td></tr>
            <tr><td>2025</td><td>TELUR AYAM</td><td>29297</td></tr>
            <tr><td>2025</td><td>DAGING SAPI</td><td>138353</td></tr>
            <tr><td>2025</td><td>GULA PASIR</td><td>18430</td></tr>
            <tr><td>2025</td><td>MINYAK GORENG</td><td>19652</td></tr>
            <tr><td>2024</td><td>BERAS IR I</td><td>14734</td></tr>
            <tr><td>2024</td><td>BERAS IR III</td><td>13517</td></tr>
            <tr><td>2024</td><td>CABAI RAWIT MERHA</td><td>59735</td></tr>
            <tr><td>2024</td><td>CABAI MERAH KERITING</td><td>53865</td></tr>
            <tr><td>2024</td><td>BAWANG MERAH</td><td>41780</td></tr>
            <tr><td>2024</td><td>BAWANG PUTIH</td><td>44106</td></tr>
            <tr><td>2024</td><td>DAGING AYAM</td><td>40525</td></tr>
            <tr><td>2024</td><td>TELUR AYAM</td><td>28631</td></tr>
            <tr><td>2024</td><td>DAGING SAPI</td><td>138588</td></tr>
            <tr><td>2024</td><td>GULA PASIR</td><td>17434</td></tr>
            <tr><td>2024</td><td>MINYAK GORENG</td><td>17033</td></tr>
          </tbody>
        </table>
        """
        soup = BeautifulSoup(raw_html, "html.parser")
    else:
        print(f"📖 Membaca berkas data dari: {file_path}")
        if file_path.endswith('.csv'):
            return pd.read_csv(file_path)
        with open(file_path, "r", encoding="utf-8") as f:
            soup = BeautifulSoup(f.read(), "html.parser")

    table = soup.find("table")
    rows = []
    for tr in table.find_all("tr"):
        cells = [c.get_text(strip=True) for c in tr.find_all(["td", "th"])]
        if cells:
            rows.append(cells)
            
    df = pd.DataFrame(rows[1:], columns=[c.strip().lower() for c in rows[0]])
    df["periode_data"] = df["periode_data"].astype(int)
    df["harga_rata_rata"] = df["harga_rata_rata"].astype(str).str.replace(",", "", regex=False).astype(float)
    df["komoditas"] = df["komoditas"].str.strip().str.upper().replace({"CABAI RAWIT MERHA": "CABAI RAWIT MERAH"})
    df = df.sort_values(["periode_data", "komoditas"]).reset_index(drop=True)
    return df

df = load_harga_pangan()
display(df.head(11))


## 1. Transformasi Matriks Pivot & Perhitungan Inflasi YoY

In [ ]:
# Membuat Matriks Pivot Komoditas x Tahun
pivot = df.pivot_table(index='komoditas', columns='periode_data', values='harga_rata_rata')
pivot.columns = [int(c) for c in pivot.columns]

# Perhitungan Inflasi YoY (%)
yoy_df = pd.DataFrame(index=pivot.index)
yoy_df['YoY_2023_2024 (%)'] = ((pivot[2024] - pivot[2023]) / pivot[2023] * 100).round(2)
yoy_df['YoY_2024_2025 (%)'] = ((pivot[2025] - pivot[2024]) / pivot[2024] * 100).round(2)

# Perhitungan CAGR 2 Tahun (%)
yoy_df['CAGR_2Y (%)'] = (((pivot[2025] / pivot[2023]) ** (1.0 / 2)) - 1.0) * 100
yoy_df['CAGR_2Y (%)'] = yoy_df['CAGR_2Y (%)'].round(2)

display(pd.concat([pivot, yoy_df], axis=1))

### 📊 Analisis Inflasi Komoditas
- **Top 3 Komoditas dengan CAGR Tertinggi (2023–2025)**:
  1. **Bawang Merah**: +12.43% per tahun (Total lonjakan: Rp 39.974 → Rp 50.531)
  2. **Minyak Goreng**: +11.61% per tahun (Total lonjakan: Rp 15.777 → Rp 19.652)
  3. **Gula Pasir**: +11.01% per tahun (Total lonjakan: Rp 14.954 → Rp 18.430)
- **Komoditas Paling Stabil / Deflasi Ringan**:
  - **Daging Sapi**: CAGR -0.69% (Rp 140.288 → Rp 138.353)
  - **Telur Ayam**: CAGR +0.50% (Rp 29.005 → Rp 29.297)

In [ ]:
# Visualisasi Heatmap Inflasi dan Lintasan Tren
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 7))

# Heatmap Matriks Harga
sns.heatmap(pivot.sort_values(by=2025, ascending=False), annot=True, fmt=',.0f', cmap='YlOrRd', ax=ax1, linewidths=1)
ax1.set_title('Matriks Harga Komoditas Pangan (IDR)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Tahun Pengamatan', fontweight='bold')

# Peringkat CAGR
cagr_sorted = yoy_df['CAGR_2Y (%)'].sort_values(ascending=True)
colors = ['#d73027' if v > 10 else '#fee08b' if v > 5 else '#1a9850' if v >= 0 else '#4575b4' for v in cagr_sorted]
ax2.barh(cagr_sorted.index, cagr_sorted.values, color=colors, height=0.6, edgecolor='#333')
ax2.axvline(0, color='black', linewidth=0.8)
ax2.set_title('Peringkat Laju Pertumbuhan Tahunan Majemuk (CAGR %)', fontsize=12, fontweight='bold')
ax2.set_xlabel('CAGR (%)', fontweight='bold')

for bar in ax2.patches:
    val = bar.get_width()
    offset = 0.2 if val >= 0 else -1.2
    ax2.text(val + offset, bar.get_y() + bar.get_height()/2, f"{val:.1f}%", va='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

---
### Final Summary
### Q&A
- **Q**: Komoditas mana yang mengalami tekanan inflasi paling persisten dalam 2 tahun terakhir?  
  **A**: Bawang Merah (+12.43% CAGR) dan Minyak Goreng (+11.61% CAGR).

### Data Analysis Key Findings
- 9 dari 11 komoditas mengalami kenaikan harga kumulatif, hanya Daging Sapi yang mengalami penurunan nominal (-1.38%).
- Beras IR I dan IR III mengalami lonjakan inflasi tertinggi pada 2023→2024 (+15.47% dan +20.27%) sebelum melandai di 2025.

### Insights or Next Steps
- Lakukan segmentasi klaster komoditas berbasis volatilitas harga untuk membedakan komoditas pangan pokok yang rentan terhadap guncangan cuaca.